#### 1A Import Cease Data

In [1]:
#Libraries
import os
import pandas as pd
print(os.getcwd())

C:\Users\user


In [2]:
def ingest_data(inputdata):
    out=pd.read_csv(inputdata)
    print(out.shape)
    return out

df_cease=ingest_data("cease.csv")
df_cease.info()

(146363, 5)
<class 'pandas.DataFrame'>
RangeIndex: 146363 entries, 0 to 146362
Data columns (total 5 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   unique_customer_identifier  146363 non-null  str  
 1   cease_placed_date           146363 non-null  str  
 2   cease_completed_date        119146 non-null  str  
 3   reason_description          146363 non-null  str  
 4   reason_description_insight  146363 non-null  str  
dtypes: str(5)
memory usage: 5.6 MB


In [3]:
# Converting date variables to datestime
df_cease['cease_placed_date']=pd.to_datetime(df_cease['cease_placed_date'])
df_cease['cease_completed_date']=pd.to_datetime(df_cease['cease_completed_date'])
df_cease.info()

<class 'pandas.DataFrame'>
RangeIndex: 146363 entries, 0 to 146362
Data columns (total 5 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   unique_customer_identifier  146363 non-null  str           
 1   cease_placed_date           146363 non-null  datetime64[us]
 2   cease_completed_date        119146 non-null  datetime64[us]
 3   reason_description          146363 non-null  str           
 4   reason_description_insight  146363 non-null  str           
dtypes: datetime64[us](2), str(3)
memory usage: 5.6 MB


In [4]:
#Getting Stats on  cease data

print("="*75)
print("Categorical Variables")
print("="*75)

for i in df_cease.select_dtypes(include=["object","category"]):
    print("="*30)
    print(f"Variable Name: {i}")
    print("Nulls: ",df_cease[i].isnull().sum())
    print(df_cease[i].describe())

print("="*75)
print("Numeric Variables")
print("="*75)

for i in df_cease.select_dtypes(exclude=["object","category"]):
    print("="*30)
    print(f"Variable Name: {i}")
    print("Nulls: ",df_cease[i].isnull().sum())
    print(df_cease[i].describe())

Categorical Variables
Variable Name: unique_customer_identifier
Nulls:  0


C:\Users\user\AppData\Local\Temp\ipykernel_18164\2954729564.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df_cease.select_dtypes(include=["object","category"]):


count                                                146363
unique                                               130934
top       f912da887651950864f3a5415b207125832896bedf17bc...
freq                                                     19
Name: unique_customer_identifier, dtype: object
Variable Name: reason_description
Nulls:  0
count        146363
unique          107
top       Not Known
freq          60010
Name: reason_description, dtype: object
Variable Name: reason_description_insight
Nulls:  0
count          146363
unique             11
top       VagueReason
freq            76055
Name: reason_description_insight, dtype: object
Numeric Variables
Variable Name: cease_placed_date
Nulls:  0
count                        146363
mean     2023-07-24 22:27:55.259457
min             2022-08-01 00:00:00
25%             2023-01-26 00:00:00
50%             2023-07-12 00:00:00
75%             2024-01-18 00:00:00
max             2024-09-01 00:00:00
Name: cease_placed_date, dtype: object
Variable

In [5]:
for i in df_cease.select_dtypes(include=["object","category"]):
    if i not in ('unique_customer_identifier'):
        print("="*30)
        print(f"Variable Name: {i}")
        print(df_cease[[i,'unique_customer_identifier']].groupby(by=i).nunique().sort_values(by='unique_customer_identifier',ascending=False).head(10))

Variable Name: reason_description


C:\Users\user\AppData\Local\Temp\ipykernel_18164\862189637.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df_cease.select_dtypes(include=["object","category"]):


                                                   unique_customer_identifier
reason_description                                                           
Not Known                                                               54645
Competitor Deals - No longer required                                   21544
Competitor Deals - Better Broadband Deal                                16569
Cease                                                                   15848
Competitor Deals - Mobile and Broadband Deal                            10042
Bereavement                                                              8427
Competitor Deals - Already Signed up for contract                        2774
Cancelled By Customer                                                    2357
Home mover - home mover process initiated                                2308
Cancelled By Provider                                                    1316
Variable Name: reason_description_insight
                      

In [6]:
#Investigating records where unique_customer_identifier duplicated
mode_val=df_cease['unique_customer_identifier'].mode()
print(mode_val[0])
df_dupes=df_cease[df_cease['unique_customer_identifier']==mode_val[0]]
#df_cease[df_cease['unique_customer_identifier'] == mode_val]

f912da887651950864f3a5415b207125832896bedf17bc85188e4aa77b5c4fad


In [7]:
df_cease_sorted=df_cease.sort_values(by=["unique_customer_identifier", "cease_completed_date"], ascending=False, na_position="first")
df_cease_sorted

df_cease_cleaned=df_cease_sorted.drop_duplicates(subset=["unique_customer_identifier", "cease_completed_date"], keep="last")
df_cease_cleaned.info()

clean=df_cease_cleaned[df_cease_cleaned['cease_completed_date'].isnull()==False]
clean

<class 'pandas.DataFrame'>
Index: 142996 entries, 89681 to 20825
Data columns (total 5 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   unique_customer_identifier  142996 non-null  str           
 1   cease_placed_date           142996 non-null  datetime64[us]
 2   cease_completed_date        119027 non-null  datetime64[us]
 3   reason_description          142996 non-null  str           
 4   reason_description_insight  142996 non-null  str           
dtypes: datetime64[us](2), str(3)
memory usage: 6.5 MB


,unique_customer_identifier,cease_placed_date,cease_completed_date,reason_description,reason_description_insight
89681,ffffe2e37966e6d685f50a6d65b2d22f4cdf193653342d...,2024-07-04,2024-08-05,Competitor Deals - No longer required,CompetitorDeals
106232,ffff339e459262da2c4e75025d75d47102fc34b7cdb24e...,2023-05-22,2023-06-07,Not Known,VagueReason
48868,fffeb3cc16c2080fcdf038b2181a1243353f32af681e27...,2023-07-19,2023-07-27,Competitor Deals - No longer required,CompetitorDeals
128383,fffe958a3aeb2e8403ace6633ddd7242b9e64c95e53638...,2023-03-10,2023-04-18,Cease,VagueReason
47622,fffe81e6ce407c50fe33da96b929dc770f97f7e7651727...,2024-02-08,2024-02-23,Not Known,VagueReason
...,...,...,...,...,...
95599,00030051141b2b1746c2f1de9e2a20e8f1c4fc79ebcddc...,2023-11-21,2023-12-21,Cease,VagueReason
45167,0002f55dbb65a594ad950925c4c3bc75bf2f61a100ac5b...,2022-08-12,2022-08-24,Competitor Deals - Already Signed up for contract,CompetitorDeals
96273,0002b09c8be6797b1d89d124939f5f1e3898df9710e356...,2023-07-24,2023-08-24,Competitor Deals - No longer required,CompetitorDeals
76234,00022d19fee89963a036dced05e68706826aab6b4013eb...,2023-01-16,2023-01-24,Bereavement,Bereavement


In [8]:
#Now writing a method for deduplication

def deduplication(df,var1,var2):
    df_cease_sorted=df_cease.sort_values(by=[var1, var2], ascending=False, na_position="first")
    df_cease_cleaned=df_cease_sorted.drop_duplicates(subset=[var1, var2], keep="last")
    deduped=df_cease_cleaned[df_cease_cleaned[var2].isnull()==False]

    return deduped

df_cease_deduped=deduplication(df_cease,"unique_customer_identifier","cease_completed_date")  
df_cease_deduped.info()

df_cease_cleaned=df_cease_deduped

<class 'pandas.DataFrame'>
Index: 119027 entries, 89681 to 20825
Data columns (total 5 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   unique_customer_identifier  119027 non-null  str           
 1   cease_placed_date           119027 non-null  datetime64[us]
 2   cease_completed_date        119027 non-null  datetime64[us]
 3   reason_description          119027 non-null  str           
 4   reason_description_insight  119027 non-null  str           
dtypes: datetime64[us](2), str(3)
memory usage: 5.4 MB


#### 1B Import Customer Info

In [9]:
#!pip install fastparquet
df_cust=pd.read_parquet("customer_info.parquet")
print(df_cust.info())
df_cust.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 3545538 entries, 0 to 3545537
Data columns (total 12 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   unique_customer_identifier  object        
 1   datevalue                   datetime64[ns]
 2   contract_status             object        
 3   contract_dd_cancels         int64         
 4   dd_cancel_60_day            int32         
 5   ooc_days                    float64       
 6   technology                  object        
 7   speed                       int32         
 8   line_speed                  float64       
 9   sales_channel               object        
 10  crm_package_name            object        
 11  tenure_days                 int32         
dtypes: datetime64[ns](1), float64(2), int32(3), int64(1), object(5)
memory usage: 284.0+ MB
None


,unique_customer_identifier,datevalue,contract_status,contract_dd_cancels,dd_cancel_60_day,ooc_days,technology,speed,line_speed,sales_channel,crm_package_name,tenure_days
0,13cf85d8849116d8d43a96ab5b77d4b3ff436c62933421...,2023-03-01,05 Newly OOC,0,0,33.0,FTTC,65,77.112,Migrated Customer,Fibre 65 (FTTC-OR),5530
1,e16582137d529fc100e04e5eec16028319d41c5df56dc0...,2023-10-01,02 In Contract,0,0,-325.0,FTTC,35,40.000,Unknown,Fibre 35 (FTTC-OR),3474
2,ffa48a8b7143996dd855d17c964311f6d0043a3070c94f...,2024-03-01,02 In Contract,0,0,-323.0,FTTC,35,40.000,Migrated Customer,Fibre 35 (FTTC-OR),4938


In [10]:
#Getting Stats on customer data
def stats(df):

    print("="*75)
    print("Categorical Variables")
    print("="*75)

    for i in df.select_dtypes(include=["object","category"]):
        print("="*30)
        print(f"Variable Name: {i}")
        print("Nulls: ",df[i].isnull().sum())
        print(df[i].describe())
   
    print("="*75)
    print("Numeric Variables")
    print("="*75)
   
    for i in df.select_dtypes(exclude=["object","category"]):
        print("="*30)
        print(f"Variable Name: {i}")
        print("Nulls: ",df[i].isnull().sum())
        print(df[i].describe())

stats(df_cust)

Categorical Variables
Variable Name: unique_customer_identifier
Nulls:  0
count                                               3545538
unique                                               202782
top       39fdde129407b5b7b4e4f056f02ba6d4c966e2f2682af2...
freq                                                     52
Name: unique_customer_identifier, dtype: object
Variable Name: contract_status
Nulls:  0
count            3545538
unique                 6
top       02 In Contract
freq             1673081
Name: contract_status, dtype: object
Variable Name: technology
Nulls:  0
count     3545538
unique          4
top          FTTC
freq      3012289
Name: technology, dtype: object
Variable Name: sales_channel
Nulls:  0
count                3545538
unique                    13
top       Online - Affiliate
freq                  885088
Name: sales_channel, dtype: object
Variable Name: crm_package_name
Nulls:  0
count                3545538
unique                    64
top       Fibre 65 (FTTC-OR)
f

In [11]:
mode_val=df_cust['unique_customer_identifier'].mode()
print(mode_val[0])

0120a0bd056bcd1356098b0d7cdd0bbde39615d9e36f1e23df998a49f445cde2


In [12]:
mode_val=df_cease['unique_customer_identifier'].mode()
print(mode_val[0])

f912da887651950864f3a5415b207125832896bedf17bc85188e4aa77b5c4fad


In [13]:
df_cust_dupes=df_cust[df_cust['unique_customer_identifier']==mode_val[0]]
df_cust_dupes.head(4)

,unique_customer_identifier,datevalue,contract_status,contract_dd_cancels,dd_cancel_60_day,ooc_days,technology,speed,line_speed,sales_channel,crm_package_name,tenure_days
2186654,f912da887651950864f3a5415b207125832896bedf17bc...,2024-03-01,01 Early Contract,0,0,-545.0,FTTC,65,79.999000,Retail,Fibre 65 (FTTC-OR),10
2452585,f912da887651950864f3a5415b207125832896bedf17bc...,2024-04-01,01 Early Contract,0,0,-514.0,FTTC,65,78.926586,Retail,Fibre 65 (FTTC-OR),41


In [14]:
for i in df_cust_dupes.select_dtypes(include=["object","category"]):
    #print(i)
    print(df_cust_dupes[[i,'tenure_days']].groupby(by=i, dropna=False).count())


                                                    tenure_days
unique_customer_identifier                                     
f912da887651950864f3a5415b207125832896bedf17bc8...            2
                   tenure_days
contract_status               
01 Early Contract            2
            tenure_days
technology             
FTTC                  2
               tenure_days
sales_channel             
Retail                   2
                    tenure_days
crm_package_name               
Fibre 65 (FTTC-OR)            2


In [15]:
df_cust[['unique_customer_identifier','contract_status']].groupby(by='contract_status', dropna=False).nunique()

,unique_customer_identifier
contract_status,
01 Early Contract,90336
02 In Contract,151139
03 Soon to be OOC,138972
04 Coming OOC,112341
05 Newly OOC,81170
06 OOC,84867


In [16]:
for i in df_cust.select_dtypes(include=["object","category"]):
    if i not in ('unique_customer_identifier'):
        print("="*30)
        print(f"Variable Name: {i}")
        print(df_cust[[i,'unique_customer_identifier']].groupby(by=i).nunique().sort_values(by='unique_customer_identifier',ascending=False).head(10))

Variable Name: contract_status
                   unique_customer_identifier
contract_status                              
02 In Contract                         151139
03 Soon to be OOC                      138972
04 Coming OOC                          112341
01 Early Contract                       90336
06 OOC                                  84867
05 Newly OOC                            81170
Variable Name: technology
            unique_customer_identifier
technology                            
FTTC                            175884
MPF                              23581
GFAST                            10084
FTTP                               887
Variable Name: sales_channel
                    unique_customer_identifier
sales_channel                                 
Online - Affiliate                       56791
Migrated Customer                        27332
Inbound                                  25947
Unknown                                  22171
Partners                      

#### 1C Calls 

In [17]:
def ingest_data(inputdata):
    out=pd.read_csv(inputdata)
    print(out.shape)
    return out

df_calls=ingest_data("calls.csv")
df_calls.info()

(628437, 5)
<class 'pandas.DataFrame'>
RangeIndex: 628437 entries, 0 to 628436
Data columns (total 5 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   unique_customer_identifier  628437 non-null  str    
 1   event_date                  628437 non-null  str    
 2   call_type                   610489 non-null  str    
 3   talk_time_seconds           628437 non-null  float64
 4   hold_time_seconds           628437 non-null  float64
dtypes: float64(2), str(3)
memory usage: 24.0 MB


In [18]:
df_calls['event_date']=pd.to_datetime(df_calls['event_date'])

In [19]:
#Getting stats on calls 
stats(df_calls)

Categorical Variables
Variable Name: unique_customer_identifier
Nulls:  0


C:\Users\user\AppData\Local\Temp\ipykernel_18164\2905256405.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df.select_dtypes(include=["object","category"]):


count                                                628437
unique                                               128833
top       1a4ce49c0157d0ce4b984e8f3ffdf82068fa51a3688cd3...
freq                                                    402
Name: unique_customer_identifier, dtype: object
Variable Name: call_type
Nulls:  17948
count     610489
unique        15
top         Tech
freq      171942
Name: call_type, dtype: object
Numeric Variables
Variable Name: event_date
Nulls:  0
count                        628437
mean     2023-06-06 09:15:55.943078
min             2022-08-01 00:00:00
25%             2022-12-12 00:00:00
50%             2023-05-02 00:00:00
75%             2023-11-11 00:00:00
max             2024-09-01 00:00:00
Name: event_date, dtype: object
Variable Name: talk_time_seconds
Nulls:  0
count    628437.000000
mean        680.093616
std         662.157413
min           0.000000
25%         237.000000
50%         504.000000
75%         911.000000
max       16017.000000
Name: tal

In [20]:
for i in df_calls.select_dtypes(include=["object","category"]):
    if i not in ('unique_customer_identifier'):
        print("="*30)
        print(f"Variable Name: {i}")
        print(df_calls[[i,'unique_customer_identifier']].groupby(by=i).nunique().sort_values(by='unique_customer_identifier',ascending=False).head(10))

Variable Name: call_type


C:\Users\user\AppData\Local\Temp\ipykernel_18164\3394993629.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df_calls.select_dtypes(include=["object","category"]):


                         unique_customer_identifier
call_type                                          
Loyalty                                       76884
CS&B                                          69128
Tech                                          53097
Customer Finance                              26124
FTTP                                          14017
Other                                          2704
Order Management                               2665
Complaints                                     1090
TTB - Sales                                     264
TTB - Customer Services                         219


#### 1D Usage data

In [21]:
#!pip install fastparquet
df_broadband=pd.read_parquet("usage.parquet")
print(df_broadband.info())
df_broadband.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 83185050 entries, 0 to 83185049
Data columns (total 4 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   unique_customer_identifier  object        
 1   calendar_date               datetime64[ns]
 2   usage_download_mbs          object        
 3   usage_upload_mbs            object        
dtypes: datetime64[ns](1), object(3)
memory usage: 2.5+ GB
None


,unique_customer_identifier,calendar_date,usage_download_mbs,usage_upload_mbs
0,9a87ea1d3811ec1b9c78d9fd12365648ba2203508545c6...,2022-09-22,9860.716,1032.873
1,d03550f4797142c2fe145fcbeb7ec247b7771b5153605d...,2022-09-22,3200.633,151.137
2,ed854191c887a386f417e64bb0814ffa157d147891b070...,2022-09-22,3474.182,106.833


In [22]:
#Customer level summary of input tables

print("df_cease_cleaned: ","Volume: ",df_cease_cleaned['unique_customer_identifier'].count(),",","Unique customers : ",df_cease_cleaned['unique_customer_identifier'].nunique())
print("df_cust","Volume: ",df_cust['unique_customer_identifier'].count(),",","Unique customers : ",df_cust['unique_customer_identifier'].nunique())
print("df_calls","Volume: ",df_calls['unique_customer_identifier'].count(),",","Unique customers : ",df_calls['unique_customer_identifier'].nunique())
print("df_broadband","Volume: ",df_broadband['unique_customer_identifier'].count(),",","Unique customers : ",df_broadband['unique_customer_identifier'].nunique())

df_cease_cleaned:  Volume:  119027 , Unique customers :  117559
df_cust Volume:  3545538 , Unique customers :  202782
df_calls Volume:  628437 , Unique customers :  128833
df_broadband Volume:  83185050 , Unique customers :  186720


#### Notes on approach
As we can see from the data summary, we have a customer base covering around 202k unique customers, with approximately 117k unique customers having completed a cease. We also have additional behavioural features available through customer call interactions and broadband usage data.
My approach is to use the customer_info dataset as the base table and create the target variable by flagging customers who completed a cease as 1 and all others as 0. To avoid data leakage, the modelling dataset will only use information available before the cease completion date. The dataset can be improved with features from the calls and broadband usage datasets using only data prior to the cease date to create realistic churn prediction model.



#### 2A Preparing cease data

In [23]:
df_cease_cleaned_sorted=df_cease_cleaned.sort_values(by=['unique_customer_identifier','cease_completed_date'],ascending=True)
print(df_cease_cleaned_sorted.shape)

df_churn_final=df_cease_cleaned_sorted.drop_duplicates(subset=['unique_customer_identifier'])
print(df_churn_final.shape)

df_churn_final['df_churn']=1
#df_churn_final.head(3)

(119027, 5)
(117559, 5)


In [24]:
df_churn_final.drop(columns=['cease_placed_date','reason_description'],inplace=True)
#df_churn_final.head(3)

#### 2B Preparing customer data

In [25]:
df_cust2=df_cust.sort_values(by=['unique_customer_identifier','datevalue'],ascending=True)
print(df_cust2.shape)

df_cust3=df_cust2.drop_duplicates(subset=['unique_customer_identifier'],keep='last')
print(df_cust3.shape)

df_cust3.head(3)
df_customer_final=df_cust3

(3545538, 12)
(202782, 12)


In [26]:
print("df_churn_final: ","Volume: ",df_churn_final['unique_customer_identifier'].count(),",","Unique customers : ",df_churn_final['unique_customer_identifier'].nunique())
print("df_customer_final","Volume: ",df_customer_final['unique_customer_identifier'].count(),",","Unique customers : ",df_customer_final['unique_customer_identifier'].nunique())

df_churn_final:  Volume:  117559 , Unique customers :  117559
df_customer_final Volume:  202782 , Unique customers :  202782


In [27]:
#Merging customer and churn data

df_combined2=df_customer_final.merge(df_churn_final, how="left", on="unique_customer_identifier")

print("df_customer_final: ","Volume: ",df_customer_final['unique_customer_identifier'].count(),",","Unique customers : ",df_customer_final['unique_customer_identifier'].nunique())
print("df_churn_final: ","Volume: ",df_churn_final['unique_customer_identifier'].count(),",","Unique customers : ",df_churn_final['unique_customer_identifier'].nunique())
print("df_combined2: ","Volume: ",df_combined2['unique_customer_identifier'].count(),",","Unique customers : ",df_combined2['unique_customer_identifier'].nunique())

df_customer_final:  Volume:  202782 , Unique customers :  202782
df_churn_final:  Volume:  117559 , Unique customers :  117559
df_combined2:  Volume:  202782 , Unique customers :  202782


In [28]:
df_combined2.info()

<class 'pandas.DataFrame'>
RangeIndex: 202782 entries, 0 to 202781
Data columns (total 15 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   unique_customer_identifier  202782 non-null  object        
 1   datevalue                   202782 non-null  datetime64[ns]
 2   contract_status             202782 non-null  object        
 3   contract_dd_cancels         202782 non-null  int64         
 4   dd_cancel_60_day            202782 non-null  int32         
 5   ooc_days                    201864 non-null  float64       
 6   technology                  202782 non-null  object        
 7   speed                       202782 non-null  int32         
 8   line_speed                  202782 non-null  float64       
 9   sales_channel               202782 non-null  object        
 10  crm_package_name            202782 non-null  object        
 11  tenure_days                 202782 non-null  int32

In [29]:
#Keep only customer info prior to cease_completed_date to avoid data leakage
df_combined2B=df_combined2[(df_combined2['cease_completed_date'].isnull()) | (df_combined2['datevalue'] < df_combined2['cease_completed_date'])]

print("df_customer_final: ","Volume: ",df_customer_final['unique_customer_identifier'].count(),",","Unique customers : ",df_customer_final['unique_customer_identifier'].nunique())
print("df_churn_final: ","Volume: ",df_churn_final['unique_customer_identifier'].count(),",","Unique customers : ",df_churn_final['unique_customer_identifier'].nunique())
print("df_combined2: ","Volume: ",df_combined2['unique_customer_identifier'].count(),",","Unique customers : ",df_combined2['unique_customer_identifier'].nunique())
print("df_combined2B: ","Volume: ",df_combined2B['unique_customer_identifier'].count(),",","Unique customers : ",df_combined2B['unique_customer_identifier'].nunique())

df_customer_final:  Volume:  202782 , Unique customers :  202782
df_churn_final:  Volume:  117559 , Unique customers :  117559
df_combined2:  Volume:  202782 , Unique customers :  202782
df_combined2B:  Volume:  200714 , Unique customers :  200714


In [30]:
for i in df_combined2B.columns:
    print("="*50)
    print(i,'unique_vals',df_combined2[i].nunique())

unique_customer_identifier unique_vals 202782
datevalue unique_vals 26
contract_status unique_vals 6
contract_dd_cancels unique_vals 2
dd_cancel_60_day unique_vals 2
ooc_days unique_vals 6045
technology unique_vals 4
speed unique_vals 9
line_speed unique_vals 70583
sales_channel unique_vals 13
crm_package_name unique_vals 63
tenure_days unique_vals 7196
cease_completed_date unique_vals 795
reason_description_insight unique_vals 11
df_churn unique_vals 1


#### 2C Preparing call data

In [31]:
df_calls_sorted=df_calls.sort_values(by=['unique_customer_identifier','event_date'],ascending=True)

df_calls_relevant=df_combined2B[["unique_customer_identifier","cease_completed_date"]].merge(df_calls_sorted, how="inner", on="unique_customer_identifier")

print("df_combined2B: ","Volume: ",df_combined2B['unique_customer_identifier'].count(),",","Unique customers : ",df_combined2B['unique_customer_identifier'].nunique())
print("df_calls_sorted: ","Volume: ",df_calls_sorted['unique_customer_identifier'].count(),",","Unique customers : ",df_calls_sorted['unique_customer_identifier'].nunique())
print("df_calls_relevant: ","Volume: ",df_calls_relevant['unique_customer_identifier'].count(),",","Unique customers : ",df_calls_relevant['unique_customer_identifier'].nunique())

print("\n")
df_calls_relevant.head()


df_combined2B:  Volume:  200714 , Unique customers :  200714
df_calls_sorted:  Volume:  628437 , Unique customers :  128833
df_calls_relevant:  Volume:  615796 , Unique customers :  127159




,unique_customer_identifier,cease_completed_date,event_date,call_type,talk_time_seconds,hold_time_seconds
0,000010b1046f4e80d74bd7ceb4ae23f167ba14cf0b2621...,2024-01-23,2023-01-18,CS&B,187.0,0.0
1,000010b1046f4e80d74bd7ceb4ae23f167ba14cf0b2621...,2024-01-23,2023-01-18,Loyalty,285.0,115.0
2,000010b1046f4e80d74bd7ceb4ae23f167ba14cf0b2621...,2024-01-23,2023-03-20,CS&B,657.0,113.0
3,000010b1046f4e80d74bd7ceb4ae23f167ba14cf0b2621...,2024-01-23,2024-01-09,Loyalty,242.0,0.0
4,00007b30558dbfc5b8e00055f5b8739d9ba006a98fb8b8...,NaT,2022-08-02,CS&B,636.0,194.0


In [32]:
df_calls_relevant2=df_calls_relevant[(df_calls_relevant['cease_completed_date'].isnull()) | (df_calls_relevant['event_date'] < df_calls_relevant['cease_completed_date'])]
print("df_calls_relevant2: ","Volume: ",df_calls_relevant2['unique_customer_identifier'].count(),",","Unique customers : ",df_calls_relevant2['unique_customer_identifier'].nunique())


df_calls_relevant2:  Volume:  615605 , Unique customers :  127147


In [33]:
#Checking correlation to potentially remove any columns 

In [34]:
#!pip install scikit-learn
from sklearn.preprocessing import LabelEncoder
enc_obj=LabelEncoder()

df_calls['call_type']=enc_obj.fit_transform(df_calls['call_type'])
df_calls.head(3)

num_vars=[]
for i in df_calls.select_dtypes(exclude=["object","category"]):
    if i not in ("cease_completed_date","event_date"):
        num_vars.append(i)

num_vars
df_calls[num_vars].corr(method='pearson')

,call_type,talk_time_seconds,hold_time_seconds
call_type,1.000000,0.117271,-0.011811
talk_time_seconds,0.117271,1.000000,0.268595
hold_time_seconds,-0.011811,0.268595,1.000000


In [35]:
df_calls[num_vars].describe()

,call_type,talk_time_seconds,hold_time_seconds
count,628437.000000,628437.000000,628437.000000
mean,5.713941,680.093616,188.788232
std,5.723217,662.157413,313.106665
min,0.000000,0.000000,0.000000
25%,0.000000,237.000000,0.000000
50%,4.000000,504.000000,15.000000
75%,14.000000,911.000000,271.000000
max,15.000000,16017.000000,9003.000000


In [36]:
# Note: Decided to drop call_type although correlation is weak to reduce dimensionality

In [37]:
df_calls_summary=df_calls[["unique_customer_identifier","talk_time_seconds","hold_time_seconds"]].groupby(by="unique_customer_identifier").sum().reset_index()
print("df_calls_summary: ","Volume: ",df_calls_summary['unique_customer_identifier'].count(),",","Unique customers : ",df_calls_summary['unique_customer_identifier'].nunique())

df_calls_summary.head(3)


df_calls_summary:  Volume:  128833 , Unique customers :  128833


,unique_customer_identifier,talk_time_seconds,hold_time_seconds
0,000010b1046f4e80d74bd7ceb4ae23f167ba14cf0b2621...,1371.0,228.0
1,00007b30558dbfc5b8e00055f5b8739d9ba006a98fb8b8...,1571.0,194.0
2,0001ed3f9d6a7919787bbca77df90a60d752d12851ad1c...,5743.0,1672.0


In [38]:
df_calls_summary[["unique_customer_identifier","talk_time_seconds","hold_time_seconds"]].describe()

,talk_time_seconds,hold_time_seconds
count,128833.000000,128833.000000
mean,3317.441898,920.893793
std,4988.820166,1478.932118
min,0.000000,0.000000
25%,774.000000,145.000000
50%,1749.000000,454.000000
75%,3936.000000,1115.000000
max,511850.000000,71105.000000


In [39]:
#Merging Model data and call data

df_combined3=df_combined2B.merge(df_calls_summary, how="left", on="unique_customer_identifier")

print("df_combined2B: ","Volume: ",df_combined2B['unique_customer_identifier'].count(),",","Unique customers : ",df_combined2B['unique_customer_identifier'].nunique())
print("df_calls_summary: ","Volume: ",df_calls_summary['unique_customer_identifier'].count(),",","Unique customers : ",df_calls_summary['unique_customer_identifier'].nunique())
print("df_combined3: ","Volume: ",df_combined3['unique_customer_identifier'].count(),",","Unique customers : ",df_combined3['unique_customer_identifier'].nunique())

df_combined2B:  Volume:  200714 , Unique customers :  200714
df_calls_summary:  Volume:  128833 , Unique customers :  128833
df_combined3:  Volume:  200714 , Unique customers :  200714


#### 2D Preparing usage data

In [40]:
df_broadband.head()

,unique_customer_identifier,calendar_date,usage_download_mbs,usage_upload_mbs
0,9a87ea1d3811ec1b9c78d9fd12365648ba2203508545c6...,2022-09-22,9860.716,1032.873
1,d03550f4797142c2fe145fcbeb7ec247b7771b5153605d...,2022-09-22,3200.633,151.137
2,ed854191c887a386f417e64bb0814ffa157d147891b070...,2022-09-22,3474.182,106.833
3,1ac8215f9e98d15b235e6baa5b4a45dafa930201348b23...,2022-09-22,16601.283,1510.906
4,17f6b51c5295d23443a9e0736dd2209b76aba82cbef303...,2022-09-22,4788.412,168.129


In [41]:
df_broadband['usage_download_mbs']=pd.to_numeric(df_broadband['usage_download_mbs'],errors="coerce").astype("int64")
df_broadband['usage_upload_mbs']=pd.to_numeric(df_broadband['usage_upload_mbs'],errors="coerce").astype("int64")

In [42]:
df_broadband.head()

,unique_customer_identifier,calendar_date,usage_download_mbs,usage_upload_mbs
0,9a87ea1d3811ec1b9c78d9fd12365648ba2203508545c6...,2022-09-22,9860,1032
1,d03550f4797142c2fe145fcbeb7ec247b7771b5153605d...,2022-09-22,3200,151
2,ed854191c887a386f417e64bb0814ffa157d147891b070...,2022-09-22,3474,106
3,1ac8215f9e98d15b235e6baa5b4a45dafa930201348b23...,2022-09-22,16601,1510
4,17f6b51c5295d23443a9e0736dd2209b76aba82cbef303...,2022-09-22,4788,168


In [43]:
df_broadband_summary=df_broadband[["unique_customer_identifier","usage_download_mbs","usage_upload_mbs"]].groupby(by="unique_customer_identifier").sum().reset_index()

In [44]:
df_broadband_summary.head()

,unique_customer_identifier,usage_download_mbs,usage_upload_mbs
0,000010b1046f4e80d74bd7ceb4ae23f167ba14cf0b2621...,19448684,504764
1,00007b30558dbfc5b8e00055f5b8739d9ba006a98fb8b8...,2689619,130894
2,0001ed3f9d6a7919787bbca77df90a60d752d12851ad1c...,6921609,153166
3,00022d19fee89963a036dced05e68706826aab6b4013eb...,58,17
4,000236cf04376bd5cb4eb6a45f8ac612f544c0ac5369d9...,6522918,258418


In [45]:
#Merging Model data and usage data

df_combined4=df_combined3.merge(df_broadband_summary, how="left", on="unique_customer_identifier")

print("df_combined3: ","Volume: ",df_combined3['unique_customer_identifier'].count(),",","Unique customers : ",df_combined3['unique_customer_identifier'].nunique())
print("df_broadband_summary: ","Volume: ",df_broadband_summary['unique_customer_identifier'].count(),",","Unique customers : ",df_broadband_summary['unique_customer_identifier'].nunique())
print("df_combined4: ","Volume: ",df_combined4['unique_customer_identifier'].count(),",","Unique customers : ",df_combined4['unique_customer_identifier'].nunique())

df_combined3:  Volume:  200714 , Unique customers :  200714
df_broadband_summary:  Volume:  186720 , Unique customers :  186720
df_combined4:  Volume:  200714 , Unique customers :  200714


In [46]:
df_combined4.info()

<class 'pandas.DataFrame'>
RangeIndex: 200714 entries, 0 to 200713
Data columns (total 19 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   unique_customer_identifier  200714 non-null  object        
 1   datevalue                   200714 non-null  datetime64[ns]
 2   contract_status             200714 non-null  object        
 3   contract_dd_cancels         200714 non-null  int64         
 4   dd_cancel_60_day            200714 non-null  int32         
 5   ooc_days                    199797 non-null  float64       
 6   technology                  200714 non-null  object        
 7   speed                       200714 non-null  int32         
 8   line_speed                  200714 non-null  float64       
 9   sales_channel               200714 non-null  object        
 10  crm_package_name            200714 non-null  object        
 11  tenure_days                 200714 non-null  int32

In [47]:
cat_vars=[]
num_vars=[]

for i in df_combined4.select_dtypes(include=["object","category"]):
    cat_vars.append(i)

for i in df_combined4.select_dtypes(exclude=["object","category"]):
    num_vars.append(i)

print("cat_vars","\n",cat_vars)
print("\n")
print("num_vars","\n",num_vars)
    

cat_vars 
 ['unique_customer_identifier', 'contract_status', 'technology', 'sales_channel', 'crm_package_name', 'reason_description_insight']


num_vars 
 ['datevalue', 'contract_dd_cancels', 'dd_cancel_60_day', 'ooc_days', 'speed', 'line_speed', 'tenure_days', 'cease_completed_date', 'df_churn', 'talk_time_seconds', 'hold_time_seconds', 'usage_download_mbs', 'usage_upload_mbs']


C:\Users\user\AppData\Local\Temp\ipykernel_18164\1159548850.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df_combined4.select_dtypes(include=["object","category"]):


In [48]:
df_model=df_combined4.drop(columns=["unique_customer_identifier","datevalue","cease_completed_date"])

cat_vars=[]
num_vars=[]

for i in df_model.select_dtypes(include=["object","category"]):
    cat_vars.append(i)

for i in df_model.select_dtypes(exclude=["object","category"]):
    num_vars.append(i)

print("cat_vars","\n",cat_vars)
print("\n")
print("num_vars","\n",num_vars)


cat_vars 
 ['contract_status', 'technology', 'sales_channel', 'crm_package_name', 'reason_description_insight']


num_vars 
 ['contract_dd_cancels', 'dd_cancel_60_day', 'ooc_days', 'speed', 'line_speed', 'tenure_days', 'df_churn', 'talk_time_seconds', 'hold_time_seconds', 'usage_download_mbs', 'usage_upload_mbs']


C:\Users\user\AppData\Local\Temp\ipykernel_18164\2868944578.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df_model.select_dtypes(include=["object","category"]):


In [49]:
for i in df_model.select_dtypes(include=["object","category"]):
    df_model[i]=enc_obj.fit_transform(df_model[i])

df_model.rename(columns={"df_churn":"target"},inplace=True)
df_model.info()


C:\Users\user\AppData\Local\Temp\ipykernel_18164\1706618842.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for i in df_model.select_dtypes(include=["object","category"]):


<class 'pandas.DataFrame'>
RangeIndex: 200714 entries, 0 to 200713
Data columns (total 16 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   contract_status             200714 non-null  int64  
 1   contract_dd_cancels         200714 non-null  int64  
 2   dd_cancel_60_day            200714 non-null  int32  
 3   ooc_days                    199797 non-null  float64
 4   technology                  200714 non-null  int64  
 5   speed                       200714 non-null  int32  
 6   line_speed                  200714 non-null  float64
 7   sales_channel               200714 non-null  int64  
 8   crm_package_name            200714 non-null  int64  
 9   tenure_days                 200714 non-null  int32  
 10  reason_description_insight  200714 non-null  int64  
 11  target                      115491 non-null  float64
 12  talk_time_seconds           127159 non-null  float64
 13  hold_time_seconds        

In [51]:
df_model.to_csv("df_model.csv",index=False)